# JEPA-TTS Supercomputer Pipeline
This notebook automates the entire pipeline: downloading the fixed codebase, installing dependencies, training from scratch, and running inference/testing on the supercomputer.

### 1. Clone Codebase & Checkout Prototype Branch

In [ ]:
# Insert your GitHub Personal Access Token here (leave blank if public)
github_token = ""

repo_url = f"https://{github_token}@github.com/OmarA32/Audio-JEPA-Arabic-TTS.git" if github_token else "https://github.com/OmarA32/Audio-JEPA-Arabic-TTS.git"
!git clone --recurse-submodules $repo_url
%cd Audio-JEPA-Arabic-TTS
!git checkout prototype/v6.0.0
!git submodule update --init --recursive


### 2. Install Dependencies
Installs all libraries directly into the notebook kernel.

In [ ]:
!pip install -r requirements.txt


### 3. Hugging Face Authentication (Optional)
If you want to automatically upload weights to Hugging Face during training, paste your Write token below. This saves it securely for the training script to use.

In [ ]:
import json

HF_TOKEN = "hf_xxxxxxxxxxxxxxxxxxxxxxxxx" # Replace with your real write token!

if HF_TOKEN.startswith("hf_") and len(HF_TOKEN) > 10:
    with open("hf_config.json", "w") as f:
        json.dump({"HF_TOKEN": HF_TOKEN}, f)
    print("Token saved! The training script will now automatically upload checkpoints.")
else:
    print("No valid token provided. Auto-upload disabled.")


### 4. Download Pre-Trained Weights
Automatically fetches the latest epoch weights from Hugging Face so you can resume seamlessly.

In [ ]:
# Change --lang to english if you want the english model
!python download_from_hf.py --lang arabic


### 5A. Run Training (From Scratch)
This deletes any existing weights and trains from scratch at Epoch 0. The dataset (`MohamedRashad/common-voice-18-arabic`) is public and will download automatically during the first epoch.

In [ ]:
# Training Arguments:
# --lang [arabic/english]       Language to train on.
# --db [nawar_halabi/common_voice/clartts/ljspeech/libritts]  Database to use.
# --val                       Add this flag to enable validation (off by default).
# --resume                      Add this to resume from latest checkpoint.
# --checkpointnum 5             Upload to Hugging Face every 5 epochs.
!python train.py --checkpointnum 5


### 5B. Resume Training / Fine-Tune
If you downloaded your saved weights from Hugging Face into the `training_logs` folder, run this cell instead! It will seamlessly resume training from the latest epoch.

In [ ]:
# Training Arguments:
# --lang [arabic/english]       Language to train on.
# --db [nawar_halabi/common_voice/clartts/ljspeech/libritts]  Database to use.
# --val                       Add this flag to enable validation (off by default).
# --resume                      Add this to resume from latest checkpoint.
# --checkpointnum 5             Upload to Hugging Face every 5 epochs.
!python train.py --resume --checkpointnum 5


### 6. Model Publishing (Upload to Hugging Face)
If you manually stopped training and want to push the latest checkpoint, use this script.

In [ ]:
# Required Arguments:
# --token       Your Hugging Face write token.
# --lang        The language model to upload (arabic or english).
!python upload_to_hf.py --token "hf_your_token_here" --lang arabic


### 7. Inference (TTS Generation)
Once training finishes, generate audio from any custom text you want.

In [ ]:
# Inference Arguments:
# --text      The text you want to synthesize.
# --output    Output WAV file name.
# --lang      [arabic/english] Language to use.
# --db        [nawar_halabi/common_voice/clartts/ljspeech/libritts] Database the model trained on.
# --vocoder   [bigvgan] Vocoder to use.
# --index     [number] Optional: Fetch exact text from dataset using --db instead of typing it
!python inference.py --text "أي نص عربي تريد" --output "my_custom_audio.wav"


### 8. Test Vocoder (Ground Truth Quality)
Tests the raw quality of the vocoder by reconstructing a ground-truth mel-spectrogram from the dataset. This bypasses the neural network.

In [ ]:
# Vocoder Test Arguments:
# --lang      [arabic/english] Language of the database.
# --db        [nawar_halabi/common_voice/clartts/ljspeech/libritts] Database to grab ground truth from.
# --index     Which audio clip index from the test dataset to reconstruct (Default: 10)
# --vocoder   [bigvgan] Vocoder to use.
# (Note: Outputs are automatically saved as test_results/ground_truth_index_10_bigvgan.wav)
!python test_vocoder_ground_truth.py --index 10 --vocoder bigvgan
